In [1]:

import numpy as np
from skimage.transform import (hough_line, hough_line_peaks, hough_circle, hough_circle_peaks)
from skimage.draw import circle_perimeter
from skimage.feature import canny
from skimage.data import astronaut
from skimage.io import imread, imsave
from skimage.color import rgb2gray, gray2rgb, label2rgb
from skimage import img_as_float
from skimage.morphology import skeletonize
from skimage import data, img_as_float
import matplotlib.pyplot as pylab
from matplotlib import cm
from skimage.filters import sobel, threshold_otsu
from skimage.feature import canny
from skimage.segmentation import felzenszwalb, slic, quickshift, watershed
from skimage.segmentation import mark_boundaries, find_boundaries
from scipy.ndimage import median_filter
import os


ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [37]:
import os
import numpy as np
import matplotlib.pyplot as plt

from skimage import io, color, exposure, filters, morphology, measure, util
from skimage.restoration import denoise_bilateral
from scipy import ndimage as ndi


INPUT_FOLDER = "wood"
OUTPUT_FOLDER = "segmented_damage"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)


def isolate_damage(image_path, save_prefix):
    img = io.imread(image_path)

    if img.shape[-1] == 4:
        img = img[:, :, :3]

    gray = color.rgb2gray(img)

    # Contrast + smoothing
    gray = exposure.equalize_adapthist(gray, clip_limit=0.02)
    gray = denoise_bilateral(
        gray,
        sigma_color=0.05,
        sigma_spatial=5,
        channel_axis=None
    )

    # Find dark defects
    dark = morphology.black_tophat(gray, morphology.disk(18))
    dark = exposure.rescale_intensity(dark, out_range=(0, 1))

    thresh = filters.threshold_otsu(dark)
    mask = dark > thresh * 1.10

    # Clean tiny noise
    mask = morphology.remove_small_objects(mask, min_size=60)
    mask = morphology.binary_closing(mask, morphology.disk(3))
    mask = ndi.binary_fill_holes(mask)

    # Remove vertical grain lines by shape filtering
    labeled = measure.label(mask)
    damage_mask = np.zeros_like(mask, dtype=bool)

    for region in measure.regionprops(labeled):
        area = region.area
        minr, minc, maxr, maxc = region.bbox

        height = maxr - minr
        width = maxc - minc

        if width == 0:
            continue

        aspect_ratio = height / width
        solidity = region.solidity
        eccentricity = region.eccentricity

        # Reject long skinny vertical grain lines
        is_vertical_line = (
            aspect_ratio > 4.0 and
            width < 20 and
            eccentricity > 0.90
        )

        # Keep larger, blob-like damaged regions
        is_damage = (
            area > 120 and
            not is_vertical_line
        )

        if is_damage:
            damage_mask[labeled == region.label] = True

    # Final smoothing
    damage_mask = morphology.binary_closing(damage_mask, morphology.disk(5))
    damage_mask = morphology.remove_small_objects(damage_mask, min_size=150)
    damage_mask = ndi.binary_fill_holes(damage_mask)

    # Overlay
    overlay = img.copy()
    overlay[damage_mask] = [255, 0, 0]
    blended = (0.45 * overlay + 0.55 * img).astype(np.uint8)

    # Save
    io.imsave(f"{OUTPUT_FOLDER}/{save_prefix}_mask.png", util.img_as_ubyte(damage_mask))
    io.imsave(f"{OUTPUT_FOLDER}/{save_prefix}_overlay.png", blended)

    return img, damage_mask, blended


# Batch process all images
valid_exts = (".png", ".jpg", ".jpeg", ".tif", ".tiff")

for filename in os.listdir(INPUT_FOLDER):
    if filename.lower().endswith(valid_exts):
        path = os.path.join(INPUT_FOLDER, filename)
        name = os.path.splitext(filename)[0]

        isolate_damage(path, name)

print("Done. Results saved in:", OUTPUT_FOLDER)

FileNotFoundError: The directory '/Users/rileymcginty/Coding Programs/School Related/CS4801-ImageProcessing/FinalProject/ImageProcessingFinal/segmented_damage' does not exist